# Phase 1: Parse and Classify Episodes

This notebook:
1. Parses all 284 transcripts
2. Uses Llama 3.1 8B to classify each episode
3. Saves structured metadata to `episodes_metadata.json`


In [1]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import json
from tqdm import tqdm

from config import TRANSCRIPTS_DIR, PROCESSED_DIR, LLM_MODEL
from parser import parse_transcript, get_all_transcripts
from classifier import classify_episode, EpisodeMetadata, save_metadata, load_metadata


In [2]:
# Check transcript directory
transcripts = list(get_all_transcripts(TRANSCRIPTS_DIR))
print(f"Found {len(transcripts)} transcripts")
print(f"\nFirst 5:")
for t in transcripts[:5]:
    print(f"  - {t.name}")


Found 284 transcripts

First 5:
  - Ada Chen Rekhi.txt
  - Adam Fishman.txt
  - Adam Grenier.txt
  - Adriel Frederick.txt
  - Aishwarya Naresh Reganti + Kiriti Badam.txt


In [3]:
# Test parsing on one transcript
sample_transcript = parse_transcript(transcripts[0])
print(f"Guest: {sample_transcript.guest_name}")
print(f"Total segments: {len(sample_transcript.segments)}")
print(f"\nFirst segment:")
print(f"  Speaker: {sample_transcript.segments[0].speaker}")
print(f"  Timestamp: {sample_transcript.segments[0].timestamp}")
print(f"  Text: {sample_transcript.segments[0].text[:300]}...")


Guest: Ada Chen Rekhi
Total segments: 202

First segment:
  Speaker: Ada Chen Rekhi
  Timestamp: 00:00:00
  Text: It's a terrible outcome to wake up one day and be late career and feel trapped because you have a certain lifestyle or a certain expectations of the people around you that you have to go work this job, but then you look at yourself in the mirror and you're not happy going in there. I think that's a ...


In [5]:
# Test LLM classification on one episode
print(f"Testing classification with {LLM_MODEL}...")
print(f"Transcript length for LLM: {len(sample_transcript.full_text_for_llm)} chars")

test_result = classify_episode(
    sample_transcript.guest_name,
    sample_transcript.full_text_for_llm,
    model=LLM_MODEL
)

print("\n" + "="*50)
print(f"Guest: {test_result.guest_name}")
print(f"Topics: {test_result.topics}")
print(f"Role: {test_result.guest_role}")
print(f"Stage: {test_result.company_stage}")
print(f"Tactical Score: {test_result.tactical_score}/10")
print(f"Contrarian Score: {test_result.contrarian_score}/10")
print(f"Summary: {test_result.one_line_summary}")
print(f"\nKey Quotes:")
for q in test_result.key_quotes:
    print(f'  - "{q}"')


Testing classification with llama3.1:8b...
Transcript length for LLM: 7964 chars

Guest: Ada Chen Rekhi
Topics: ['Career development and growth', 'Product sense and decision-making', 'Curiosity loops for contextual advice']
Role: executive coach / founder
Stage: early-stage
Tactical Score: 7/10
Contrarian Score: 5/10
Summary: Ada Chen Rekhi discusses her approach to career development, product sense, and decision-making through the use of curiosity loops.

Key Quotes:
  - "A curiosity loop is essentially going to a whole bunch of people and asking them for input in a structured way."
  - "Curiosity loops really fights the fact that there's a lot of bad advice out there. It's not bad because it's not well-intentioned, but it's bad because it's not contextual."
  - "A good question is specific, solicits rationale, and isn't biased."


In [6]:
# Process all transcripts with checkpointing
OUTPUT_FILE = PROCESSED_DIR / "episodes_metadata.json"
CHECKPOINT_FILE = PROCESSED_DIR / "episodes_metadata_checkpoint.json"

# Load existing progress if any
processed = {}
if CHECKPOINT_FILE.exists():
    existing = load_metadata(str(CHECKPOINT_FILE))
    processed = {m.guest_name: m for m in existing}
    print(f"Loaded {len(processed)} previously processed episodes")

# Filter out already processed
to_process = [t for t in transcripts if parse_transcript(t).guest_name not in processed]
print(f"Remaining to process: {len(to_process)}")


Remaining to process: 284


In [7]:
# Main processing loop with progress bar and error handling
errors = []

for filepath in tqdm(to_process, desc="Classifying episodes"):
    try:
        # Parse transcript
        parsed = parse_transcript(filepath)
        
        # Skip if already processed
        if parsed.guest_name in processed:
            continue
        
        # Classify with LLM
        metadata = classify_episode(
            parsed.guest_name,
            parsed.full_text_for_llm,
            model=LLM_MODEL
        )
        
        processed[parsed.guest_name] = metadata
        
        # Checkpoint every 10 episodes
        if len(processed) % 10 == 0:
            save_metadata(list(processed.values()), str(CHECKPOINT_FILE))
            
    except Exception as e:
        errors.append((filepath.name, str(e)))
        print(f"\nError processing {filepath.name}: {e}")
        continue

# Final save
save_metadata(list(processed.values()), str(OUTPUT_FILE))
print(f"\nCompleted! Processed {len(processed)} episodes")
print(f"Saved to: {OUTPUT_FILE}")

if errors:
    print(f"\nErrors ({len(errors)}):")
    for name, err in errors:
        print(f"  - {name}: {err[:100]}")


Classifying episodes:   3%|▎         | 9/284 [53:21<15:40:30, 205.20s/it]


Error processing Alexander Embiricos.txt: Failed to parse LLM response for Alexander Embiricos: Expecting property name enclosed in double quotes: line 9 column 23 (char 181)
Response: {
  "guest_name": "Alexander Embiricos",
  "topics": [
    "Codex",
    "OpenAI's coding agent",
    "AGI timelines",
    "product development at OpenAI"
  ],
  "guest_role": "PM", // Product Manager
  "company_stage": "growth",
  "tactical_score": 8,
  "contrarian_score": 4,
  "key_quotes": [
    "Codex is just the beginning of a software engineering teammate.",
    "The current underappreciated limiting factor is literally human typing speed or human multitasking speed.",
    "We have a Codex code review that's catching a lot of mistakes."
  ],
  "one_line_summary": "Alexander Embiricos discusses his experience working on Codex at OpenAI, its rapid growth, and the future of AGI"
}


Classifying episodes: 100%|██████████| 284/284 [8:47:06<00:00, 111.36s/it]  


Completed! Processed 283 episodes
Saved to: /Users/nanditakrishnan/llm/lenny-podcast/notebooks/../data/processed/episodes_metadata.json

Errors (1):
  - Alexander Embiricos.txt: Failed to parse LLM response for Alexander Embiricos: Expecting property name enclosed in double quo


In [8]:
# Quick stats on the classified data
from collections import Counter

all_metadata = load_metadata(str(OUTPUT_FILE))

print(f"Total episodes classified: {len(all_metadata)}")

# Topic distribution
all_topics = [t for m in all_metadata for t in m.topics]
topic_counts = Counter(all_topics)
print(f"\nTop 15 Topics:")
for topic, count in topic_counts.most_common(15):
    print(f"  {topic}: {count}")

# Role distribution
role_counts = Counter(m.guest_role for m in all_metadata)
print(f"\nGuest Roles:")
for role, count in role_counts.most_common():
    print(f"  {role}: {count}")

# Score distributions
tactical_avg = sum(m.tactical_score for m in all_metadata) / len(all_metadata)
contrarian_avg = sum(m.contrarian_score for m in all_metadata) / len(all_metadata)
print(f"\nAverage Tactical Score: {tactical_avg:.1f}/10")
print(f"Average Contrarian Score: {contrarian_avg:.1f}/10")

# Most contrarian episodes
print(f"\nMost Contrarian Episodes:")
for m in sorted(all_metadata, key=lambda x: x.contrarian_score, reverse=True)[:5]:
    print(f"  {m.guest_name} ({m.contrarian_score}/10): {m.one_line_summary[:60]}...")


Total episodes classified: 283

Top 15 Topics:
  product-market fit: 87
  growth: 53
  leadership: 42
  hiring: 40
  AI: 28
  product management: 24
  product development: 8
  company culture: 7
  innovation: 6
  productivity: 5
  team building: 4
  communication skills: 4
  product strategy: 4
  product-led growth: 4
  career development: 4

Guest Roles:
  founder/PM: 34
  founder: 33
  founder / PM: 29
  exec: 28
  founder/CEO: 28
  PM: 8
  executive coach: 6
  founder / PM / designer: 5
  coach: 5
  VC: 3
  founder/exec: 3
  researcher: 3
  PM/Founder: 2
  founder/PM/exec: 2
  consultant: 2
  VP of Product: 2
  founder/PM/designer: 2
  growth expert: 2
  head of growth: 2
  author/consultant: 2
  CPO (Chief Product Officer): 2
  VP of product: 2
  executive coach / founder: 1
  researcher/exec/teacher: 1
  consultant/strategy expert: 1
  founder / PM / designer (author and special partner at First Round Capital): 1
  VP of product and head of growth: 1
  founder / PM (product market